# 06 · Train Plate Detector (yolo_plate.pt)

> **OWNER:** Member B (M4 · hazards + plates)
> **PREREQUISITES:** `05_prepare_plates.ipynb` complete, AND its decision was
> `train_from_scratch` (see the check in Step 0 below — this notebook skips
> itself if 05 found a usable pretrained model).
> **EXPECTED RUNTIME:** ~40 minutes on a T4.
> **OUTPUTS:** `models/yolo_plate.pt`, `models/yolo_plate.onnx`,
> `models/MODEL_CARD_plate.md` — with detection mAP and end-to-end OCR
> accuracy reported **separately**.

**Next notebook:** `07_evaluate_all.ipynb`.

In [ ]:
# Cell 1/3 — minimal bootstrap (Colab vs local). No repo imports yet: on a
# fresh Colab runtime nothing has been cloned, so this cell is deliberately
# self-contained and only prepares sys.path so `common/` becomes importable.
import subprocess
import sys
from pathlib import Path


def _in_colab() -> bool:
    try:
        import google.colab  # noqa: F401

        return True
    except ImportError:
        return False


IN_COLAB = _in_colab()
REPO_URL = "https://github.com/ad8thya/SmartIndiaHackathon.git"

if IN_COLAB:
    REPO_ROOT = Path("/content/SmartIndiaHackathon")
    if not REPO_ROOT.exists():
        print(f"cloning {REPO_URL} -> {REPO_ROOT}")
        subprocess.run(["git", "clone", REPO_URL, str(REPO_ROOT)], check=True)
    else:
        print(f"{REPO_ROOT} already present locally on this runtime")
else:
    _here = Path.cwd().resolve()
    _candidates = [c for c in (_here, *_here.parents) if (c / "pyproject.toml").exists() and (c / "notebooks").exists()]
    if not _candidates:
        raise RuntimeError(
            "Could not find the repo root (looked for pyproject.toml + notebooks/ "
            f"walking up from {_here}). Run this notebook from inside the repo checkout."
        )
    REPO_ROOT = _candidates[0]

for _p in (str(REPO_ROOT), str(REPO_ROOT / "notebooks")):
    if _p not in sys.path:
        sys.path.insert(0, _p)

print(f"Colab: {IN_COLAB}")
print(f"repo root: {REPO_ROOT}")

In [ ]:
# Cell 2/3 — install the ML extras. Quiet; ~60-90s on a fresh Colab runtime,
# near-instant if already installed (pip no-ops on a satisfied requirement).
import subprocess
import sys

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-e", f"{REPO_ROOT}[ml]"],
    check=True,
)
print("ml extras installed")

In [ ]:
# Cell 3/3 — full environment setup (Drive mount + DATA_ROOT/MODEL_ROOT) and
# an import check: torch version + CUDA availability, so a broken environment
# fails here, not forty minutes into a training run.
from common import colab as colab_mod

env = colab_mod.setup_environment()
REPO_ROOT, DATA_ROOT, MODEL_ROOT = env["repo_root"], env["data_root"], env["model_root"]

import ultralytics

print(f"ultralytics {ultralytics.__version__}")
gpu_info = colab_mod.gpu_report()

## Step 0 — skip check

Reads the decision `05_prepare_plates.ipynb` wrote. If a pretrained model was already selected, this notebook has nothing to do.

In [ ]:
import json
import sys

decision_path = MODEL_ROOT / "plate_decision.json"
if not decision_path.exists():
    raise FileNotFoundError(f"{decision_path} not found — run 05_prepare_plates.ipynb through Step 1c first.")

decision = json.loads(decision_path.read_text())
print(f"decision from 05: {decision}")

if decision["source"] == "pretrained":
    print()
    print("=" * 60)
    print("SKIPPING — 05_prepare_plates.ipynb already selected a pretrained")
    print(f"plate detector ({decision['pretrained_source']}), copied to models/yolo_plate.pt.")
    print("Nothing to train here. Continue to 07_evaluate_all.ipynb.")
    print("=" * 60)
    SKIP_TRAINING = True
else:
    print("no pretrained model was usable — training from scratch below.")
    SKIP_TRAINING = False

## Step 1 — Assert data.yaml matches the frozen indices (skipped if Step 0 skipped)

In [ ]:
import yaml

from common import constants

if not SKIP_TRAINING:
    DATA_YAML_PATH = DATA_ROOT / "plates_prepared" / "data.yaml"
    with open(DATA_YAML_PATH) as f:
        data_yaml = yaml.safe_load(f)
    constants.assert_class_order(data_yaml["names"], constants.PLATE_CLASSES, "plate")

## Step 2 — Train

`yolo11n.pt`, single class, imgsz 640, epochs 60.

In [ ]:
import random

import numpy as np
import torch
from ultralytics import YOLO

if not SKIP_TRAINING:
    SEED = 42
    random.seed(SEED)
    np.random.seed(SEED)
    torch.manual_seed(SEED)

    RUNS_DIR = MODEL_ROOT / "runs" / "plate"
    plate_model = YOLO("yolo11n.pt")
    plate_results = plate_model.train(
        data=str(DATA_YAML_PATH),
        imgsz=640,
        epochs=60,
        batch=-1,
        optimizer="AdamW",
        lr0=1e-3,
        patience=20,
        seed=SEED,
        mosaic=1.0,
        fliplr=0.0,  # a plate's characters are not mirror-symmetric — a flipped plate is unreadable AND unrealistic
        project=str(RUNS_DIR),
        name="plate",
        exist_ok=True,
    )
    print(f"run saved to {plate_results.save_dir}")

## Step 3 — Two-stage evaluation: detection mAP vs end-to-end OCR accuracy, SEPARATELY

These are different numbers and conflating them overstates the system.
Detection mAP will look good; OCR accuracy will not — report both, plainly labelled.

In [ ]:
from common import evaluate

if not SKIP_TRAINING:
    detection_table = evaluate.per_class_table(plate_model, str(DATA_YAML_PATH), split="test")
    print("STAGE 1 — plate DETECTION mAP:")
    print(detection_table.to_string(index=False))

In [ ]:
import re

import cv2
from paddleocr import PaddleOCR

INDIAN_PLATE_REGEX = re.compile(r"^[A-Z]{2}[0-9]{1,2}[A-Z]{1,3}[0-9]{4}$")


def _normalise(text: str) -> str:
    return "".join(text.split()).upper()


if not SKIP_TRAINING:
    ocr = PaddleOCR(use_angle_cls=True, lang="en")
    TEST_IMAGES_DIR = DATA_ROOT / "plates_prepared" / "images" / "test"
    TEST_LABELS_DIR = DATA_ROOT / "plates_prepared" / "labels" / "test"

    # NOTE: this measures end-to-end string accuracy against each image's OWN OCR
    # read as a stand-in ground truth is NOT valid — you need real ground-truth
    # plate strings to score exact-match accuracy. If you have a small hand-labelled
    # set of (image, true_plate_string) pairs, load them into GROUND_TRUTH below.
    GROUND_TRUTH: dict[str, str] = {}  # {"image_stem": "TN09BX4412", ...} — fill in by hand

    if not GROUND_TRUTH:
        print("GROUND_TRUTH is empty — hand-label a handful of test images with their true plate")
        print("string and fill in the dict above to get a real exact-match accuracy number.")
        print("Running detection + OCR below anyway so you can see raw output and start labelling from it.")

    n_exact, n_char_correct, n_char_total, n_scored = 0, 0, 0, 0
    for stem, true_plate in GROUND_TRUTH.items():
        image_path = next(TEST_IMAGES_DIR.glob(f"{stem}.*"), None)
        if image_path is None:
            continue
        det = plate_model.predict(source=str(image_path), conf=0.25, verbose=False)[0]
        if len(det.boxes) == 0:
            continue
        box = det.boxes[0].xyxy[0].tolist()
        img = cv2.imread(str(image_path))
        x1, y1, x2, y2 = (int(v) for v in box)
        crop = img[y1:y2, x1:x2]
        result = ocr.ocr(crop, cls=True)
        texts = [line[1][0] for page in (result or []) for line in (page or [])]
        predicted = _normalise(max(texts, key=len)) if texts else ""
        true_norm = _normalise(true_plate)

        n_scored += 1
        n_exact += predicted == true_norm
        for a, b in zip(predicted, true_norm):
            n_char_total += 1
            n_char_correct += a == b
        n_char_total += abs(len(predicted) - len(true_norm))  # length mismatch penalises char accuracy too

    print()
    print("STAGE 2 — end-to-end OCR string accuracy (SEPARATE from detection mAP above):")
    if n_scored:
        print(f"  exact-match accuracy: {n_exact}/{n_scored} = {n_exact / n_scored:.1%}")
        print(f"  character-level accuracy: {n_char_correct}/{max(n_char_total, 1)} = {n_char_correct / max(n_char_total, 1):.1%}")
    else:
        print("  no scored examples — fill in GROUND_TRUTH above with real (image, plate) pairs.")
    print()
    print("Expect this number to be materially lower than the detection mAP above.")
    print("Mono-camera OCR at bus speed on Indian roads is genuinely hard — that gap is the honest result, not a bug.")

## Step 4 — Export + model card

In [ ]:
import shutil
from pathlib import Path

from common import export, model_card

if not SKIP_TRAINING:
    PLATE_PT_PATH = MODEL_ROOT / constants.MODEL_FILES["plate"]
    shutil.copy2(Path(plate_model.trainer.best), PLATE_PT_PATH)
    onnx_path = export.export_onnx(PLATE_PT_PATH, imgsz=640, opset=12)
    latency = export.benchmark_latency(PLATE_PT_PATH, onnx_path, imgsz=640)
    export.print_latency_table(latency)

    ocr_summary = (
        f"exact-match: {n_exact}/{n_scored} ({n_exact / n_scored:.1%})" if n_scored
        else "not scored — GROUND_TRUTH was empty, see Step 3"
    )
    model_card.render_model_card(
        model_name="yolo_plate.pt — Plate Detector",
        owner="M4",
        base_weights="yolo11n.pt",
        dataset="Roboflow Universe Indian number-plate dataset (single class)",
        dataset_size={"train": len(list((DATA_ROOT / "plates_prepared" / "images" / "train").iterdir())),
                      "val": len(list((DATA_ROOT / "plates_prepared" / "images" / "val").iterdir())),
                      "test": len(list((DATA_ROOT / "plates_prepared" / "images" / "test").iterdir()))},
        class_names=constants.PLATE_CLASSES,
        hyperparameters={"imgsz": 640, "epochs": 60, "optimizer": "AdamW", "lr0": 1e-3, "patience": 20, "seed": SEED},
        metrics_table_md=(
            "**Stage 1 — detection:**\n\n" + detection_table.to_markdown(index=False) +
            f"\n\n**Stage 2 — end-to-end OCR string accuracy (separate metric):** {ocr_summary}"
        ),
        latency=latency,
        caveats=[
            "Detection mAP and OCR string accuracy are DIFFERENT numbers, reported separately "
            "on purpose — quoting detection mAP alone overstates the system's real accuracy.",
            "Mono-camera plate OCR at bus speed on Indian roads is genuinely hard; expect the "
            "OCR number to be materially lower than detection mAP.",
        ],
        output_path=MODEL_ROOT / "MODEL_CARD_plate.md",
    )

---
### What this notebook produced
- If it ran: `models/yolo_plate.pt`, `models/yolo_plate.onnx`, `models/MODEL_CARD_plate.md`
  with detection mAP and OCR accuracy reported separately.
- If it skipped: nothing new — 05 already produced everything this notebook would have.

### Next
`07_evaluate_all.ipynb`.